In [1]:
# -*- coding: utf-8 -*-
"""mental_health_langgraph.ipynb

LangGraph refactor of the Mental Health Chatbot.
Original file: multimodal (2).ipynb
"""

# =====================================================
# INSTALL DEPENDENCIES
# =====================================================

%pip install langchain langchain-openai openai python-dotenv
%pip install langchain-community
%pip install pypdf
%pip install chromadb
%pip install langchain-chroma
%pip install langgraph
%pip install langchain-huggingface
%pip install rank-bm25
%pip install sentence-transformers


# =====================================================
# IMPORTS
# =====================================================
import os
import re
import json
import hashlib
import sqlite3
from time import perf_counter
from datetime import datetime
from typing import Optional

from dotenv import load_dotenv

try:
    from rank_bm25 import BM25Okapi
except Exception:
    BM25Okapi = None

try:
    from sentence_transformers import CrossEncoder
except Exception:
    CrossEncoder = None

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver
from typing_extensions import TypedDict

# =====================================================
# 1. LOAD ENVIRONMENT
# =====================================================
load_dotenv()

os.environ["OPENROUTER_API_KEY"] = "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"

if "OPENROUTER_API_KEY" not in os.environ:
    raise ValueError("OPENROUTER_API_KEY not set")

# =====================================================
# 2. SYMPTOM SCHEMA
# =====================================================
SYMPTOM_FIELDS = [
    "mood",
    "anxiety",
    "stress",
    "sleep",
    "energy",
    "appetite",
    "duration",
    "severity"
]

MAX_FOLLOWUPS = 6

# Retrieval and quality gates
PRE_CRITIC_PASS_SCORE = 7
PRE_CRITIC_MAX_RETRIES = 2
RETRIEVAL_K_DENSE = 6
RETRIEVAL_K_SPARSE = 8
FUSION_TOP_K = 8
FINAL_CONTEXT_TOP_K = 4
RRF_K = 60
QUERY_VARIANT_LIMIT = 4
QUERY_ALT_LIMIT = 4

# =====================================================
# 3. LLM SETUP (OpenRouter)
# =====================================================
compounder_llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.3,
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1"
)

doctor_llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.2,
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1"
)

summary_llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.4,
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
)

# =====================================================
# 4. PROMPTS
# =====================================================
compounder_prompt = ChatPromptTemplate.from_template("""You are a calm, empathetic mental health intake assistant.

Your role is to carefully understand the user's message and extract ALL clinically relevant mental health information mentioned, without restricting yourself to predefined symptom categories.

Follow these rules strictly:

Do NOT diagnose

Do NOT suggest medication

Do NOT assume facts not stated

Only extract what is explicitly mentioned or clearly implied

If something is not mentioned, mark it as "unknown"

Return ONLY valid JSON

No explanations

No markdown

You must perform TWO steps internally:

Step 1 - Open Clinical Extraction
Extract all relevant mental-health-related details from the user's message, including:

emotional states

behavioral changes

cognitive symptoms

physical symptoms related to mental health

stressors or triggers

coping behaviors

functional impact

risk indicators (if any)

time references

Step 2 - Critical Information Gap Detection
Based on what the user shared, identify important intake information that is missing and may be clinically relevant.

Return the result in the following JSON structure:

{{
"clinical_summary": {{
"emotional_symptoms": [],
"cognitive_symptoms": [],
"behavioral_symptoms": [],
"physical_symptoms": [],
"stressors_or_triggers": [],
"functional_impact": [],
"risk_indicators": [],
"coping_behaviors": [],
"timeline_information": []
}},
"structured_fields": {{
"mood": "",
"anxiety": "",
"stress": "",
"sleep": "",
"energy": "",
"appetite": "",
"duration": "",
"severity": ""
}},
"missing_critical_information": []
}}

If a structured field is not mentioned, set it to "unknown".
If no items exist for a list, return an empty array.
User message:
{input}
""")

followup_prompt = ChatPromptTemplate.from_template("""
You are a calm, empathetic mental health intake assistant.

Your job is ONLY to gather missing or unclear information
that can be stored in the structured symptom fields.

You may ask a follow-up question ONLY if it will help clarify
or fill ONE of the following fields:
mood, anxiety, stress, sleep, energy, appetite, duration, severity

IMPORTANT CONSTRAINTS:
- DO NOT ask "why" questions
- DO NOT ask for explanations, causes, or reasons
- DO NOT ask questions that cannot update a symptom field
- DO NOT repeat a question that has already been answered
- DO NOT ask about a field that is already known

Decision rules:
- If at least the core fields (mood OR anxiety OR stress)
  AND duration AND severity are known -> respond EXACTLY with:
  NO_FOLLOWUP_NEEDED
- Otherwise, ask ONE gentle, natural follow-up question
  that directly targets ONE missing or unclear field

Tone:
- Empathetic
- Simple
- Non-judgmental
- Conversational

Collected symptoms (JSON):
{symptoms}

Last follow-up question asked:
{last_question}

Conversation summary so far:
{conversation_summary}
""")

summary_prompt = ChatPromptTemplate.from_template("""
You are a helpful, empathetic assistant.

You will be given a JSON output produced by a "Doctor Agent".
Your job is to explain it to the user in a simple, short, friendly message.

Rules:
- Do NOT mention JSON, system, model, agents, databases
- Do NOT give medication dosages
- Do NOT sound robotic
- Keep it short (6-10 lines max)
- Use bullet points
- If red_flags are present, mention them clearly and gently
- End with a disclaimer that this is not a medical diagnosis and suggest professional consultation when needed.

Doctor output JSON:
{doctor_output}
""")

doctor_prompt = ChatPromptTemplate.from_template("""
You are a medical-support AI for mental health.

You are NOT a doctor.
You do NOT give diagnoses or medication.
You provide educational guidance based ONLY on the provided medical reference.

If the user shows signs of:
- self harm
- suicidal thoughts
- inability to function
You MUST advise contacting a mental health professional immediately.

You are given:

SYMPTOMS:
{symptoms}

USER HISTORY:
{history}

MEDICAL KNOWLEDGE (from trusted sources):
{context}

Your job:
1. Identify which mental health conditions from the knowledge best match the symptoms
2. Explain in simple, supportive language
3. Suggest safe actions (journaling, breathing, therapy, lifestyle)
4. If serious, recommend seeing a professional

Rules:
- Use ONLY the medical knowledge above
- Do NOT invent conditions
- Do NOT provide medication or dosage

Answer:
""")

pre_critic_prompt = ChatPromptTemplate.from_template("""
You are a retrieval quality evaluator.

Given:
- user symptoms JSON
- retrieved context chunks

Score sufficiency from 0 to 10.
Set sufficient=true only if context is strong enough for safe, useful guidance.
If insufficient, produce a refined retrieval query.

Return JSON only:
{{
  "score": 0,
  "sufficient": false,
  "reason": "",
  "refinement_query": ""
}}

SYMPTOMS:
{symptoms}

RETRIEVED_CONTEXT:
{context}
""")

query_generator_prompt = ChatPromptTemplate.from_template("""
You generate retrieval queries for a mental-health knowledge base.

Input:
- current structured symptoms JSON
- latest user message
- conversation summary

Requirements:
- Return JSON only
- No markdown
- Keep each query short and retrieval-friendly
- Preserve important clinical meaning
- Include one strong primary query and up to 4 alternates

Return format:
{{
  "primary_query": "",
  "alternate_queries": []
}}

STRUCTURED_SYMPTOMS:
{symptoms}

USER_TEXT:
{user_text}

CONVERSATION_SUMMARY:
{conversation_summary}
""")

# =====================================================
# 5. CHAINS
# =====================================================
compounder_chain = compounder_prompt | compounder_llm
followup_chain = followup_prompt | compounder_llm
synopsis_chain = summary_prompt | summary_llm | StrOutputParser()
pre_critic_chain = pre_critic_prompt | compounder_llm | StrOutputParser()
doctor_generation_chain = doctor_prompt | doctor_llm | StrOutputParser()
query_generator_chain = query_generator_prompt | compounder_llm | StrOutputParser()

# =====================================================
# 6. SQLITE SETUP
# =====================================================
conn = sqlite3.connect("compounder_memory.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS compounder_records (
    user_id INTEGER,
    timestamp TEXT,
    symptoms TEXT
)
""")
conn.commit()

# =====================================================
# 7. VECTOR DB SETUP (build knowledge base once)
# =====================================================
all_docs = []
for file in os.listdir("mentaldocs"):
    if file.endswith(".pdf"):
        path = os.path.join("mentaldocs", file)
        reader = PdfReader(path)
        for page_idx, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            if text.strip():
                all_docs.append(
                    Document(
                        page_content=text,
                        metadata={"source": path, "page": page_idx},
                    )
                )

def split_documents_simple(docs, chunk_size=800, chunk_overlap=100):
    split_docs = []
    step = max(1, chunk_size - chunk_overlap)
    for doc in docs:
        text = (doc.page_content or "").strip()
        if not text:
            continue
        for start in range(0, len(text), step):
            end = start + chunk_size
            chunk_text = text[start:end].strip()
            if not chunk_text:
                continue
            split_docs.append(
                Document(
                    page_content=chunk_text,
                    metadata=dict(doc.metadata or {}),
                )
            )
            if end >= len(text):
                break
    return split_docs

chunks = split_documents_simple(all_docs, chunk_size=800, chunk_overlap=100)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="doctor_kb",
    collection_metadata={"hnsw:space": "cosine"}
)

retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": RETRIEVAL_K_DENSE,
    }
)

# Sparse index (BM25) over the same chunks
chunk_texts = [d.page_content for d in chunks]

def tokenize_for_sparse(text: str) -> list:
    return re.findall(r"[a-z0-9]+", (text or "").lower())

tokenized_chunks = [tokenize_for_sparse(t) for t in chunk_texts]
if BM25Okapi is not None:
    bm25 = BM25Okapi(tokenized_chunks)
else:
    bm25 = None

# Global caches and lazy model
cross_encoder = None
retrieval_cache_dense = {}
retrieval_cache_sparse = {}
retrieval_cache_hybrid = {}

# =====================================================
# 8. HELPER FUNCTIONS
# =====================================================
def save_symptoms(user_id: int, symptoms: dict):
    cursor.execute(
        "INSERT INTO compounder_records VALUES (?, ?, ?)",
        (user_id, datetime.now().isoformat(), json.dumps(symptoms))
    )
    conn.commit()


def merge_symptoms(old: dict, new: dict) -> dict:
    merged = old.copy()
    for key, value in new.items():
        if value != "unknown":
            merged[key] = value
    return merged


def extract_symptoms(user_text: str) -> dict:
    raw = compounder_chain.invoke({"input": user_text}).content
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        raise ValueError("Symptom extraction failed (invalid JSON)")


def get_dynamic_followup(symptoms: dict, last_question: Optional[str], conversation_summary: str) -> Optional[str]:
    response = followup_chain.invoke({
        "symptoms": json.dumps(symptoms),
        "last_question": last_question if last_question else "None",
        "conversation_summary": conversation_summary if conversation_summary else "None"
    }).content.strip()

    if response == "NO_FOLLOWUP_NEEDED":
        return None
    return response


def get_user_history(user_id: int) -> str:
    conn2 = sqlite3.connect("compounder_memory.db")
    c = conn2.cursor()
    c.execute("SELECT timestamp, symptoms FROM compounder_records WHERE user_id=?", (user_id,))
    rows = c.fetchall()
    conn2.close()

    if not rows:
        return "No previous records."

    summary = ""
    for t, s in rows[-5:]:
        summary += f"{t}: {s}\n"
    return summary


def update_conversation_summary(summary: list, extracted: dict) -> list:
    summary = summary.copy()
    for key, value in extracted.items():
        if value != "unknown":
            existing_keys = [line.split(" ")[0] for line in summary]
            if key not in existing_keys:
                summary.append(f"{key} {value}")
            else:
                for i, line in enumerate(summary):
                    if line.startswith(key):
                        summary[i] = f"{key} {value}"
                        break
    return summary


def optimize_query(symptoms_json: str) -> str:
    try:
        data = json.loads(symptoms_json)
    except Exception:
        return symptoms_json

    parts = []
    for k, v in data.items():
        if v != "unknown":
            parts.append(f"{k}: {v}")

    if not parts:
        return symptoms_json

    return "mental health symptoms " + ", ".join(parts)


def normalize_query_list(items, max_items=QUERY_VARIANT_LIMIT):
    out = []
    seen = set()
    for item in items:
        if not isinstance(item, str):
            continue
        q = " ".join(item.strip().split())
        if not q:
            continue
        key = q.lower()
        if key in seen:
            continue
        seen.add(key)
        out.append(q)
        if len(out) >= max_items:
            break
    return out


def generate_query_variants(symptoms: dict, user_text: str, conversation_summary: list):
    baseline = optimize_query(json.dumps(symptoms))
    fallback = {
        "primary_query": baseline,
        "alternate_queries": [],
    }

    raw = query_generator_chain.invoke({
        "symptoms": json.dumps(symptoms),
        "user_text": user_text or "",
        "conversation_summary": "; ".join(conversation_summary or []),
    })

    parsed = safe_json(raw, fallback)

    primary = (parsed.get("primary_query") or "").strip()
    if not primary:
        primary = baseline

    alternates = parsed.get("alternate_queries") or []
    if isinstance(alternates, str):
        alternates = [alternates]
    if not isinstance(alternates, list):
        alternates = []
    alternates = alternates[:QUERY_ALT_LIMIT]

    variants = normalize_query_list([primary, baseline] + alternates, QUERY_VARIANT_LIMIT)
    if not variants:
        variants = [baseline]

    return {
        "primary_query": variants[0],
        "query_variants": variants,
        "query_generation_raw": parsed,
    }


def doc_key(doc) -> str:
    meta = getattr(doc, "metadata", {}) or {}
    source = str(meta.get("source", ""))
    page = str(meta.get("page", ""))
    body = getattr(doc, "page_content", "") or ""
    digest = hashlib.md5(body.encode("utf-8", errors="ignore")).hexdigest()
    return f"{source}|{page}|{digest}"


def retrieve_dense(query: str):
    if query in retrieval_cache_dense:
        return retrieval_cache_dense[query]
    docs = retriever.invoke(query)
    docs = docs[:RETRIEVAL_K_DENSE]
    retrieval_cache_dense[query] = docs
    return docs


def retrieve_sparse(query: str):
    if query in retrieval_cache_sparse:
        return retrieval_cache_sparse[query]
    if bm25 is None:
        return []
    q_tokens = tokenize_for_sparse(query)
    scores = bm25.get_scores(q_tokens)
    ranked_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:RETRIEVAL_K_SPARSE]
    docs = [chunks[i] for i in ranked_idx]
    retrieval_cache_sparse[query] = docs
    return docs


def reciprocal_rank_fusion(dense_docs, sparse_docs, top_k=FUSION_TOP_K):
    score_map = {}
    doc_map = {}

    for rank, d in enumerate(dense_docs, start=1):
        k = doc_key(d)
        score_map[k] = score_map.get(k, 0.0) + 1.0 / (RRF_K + rank)
        doc_map[k] = d

    for rank, d in enumerate(sparse_docs, start=1):
        k = doc_key(d)
        score_map[k] = score_map.get(k, 0.0) + 1.0 / (RRF_K + rank)
        doc_map[k] = d

    ranked = sorted(score_map.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [doc_map[k] for k, _ in ranked]


def reciprocal_rank_fusion_multi(dense_ranked_lists, sparse_ranked_lists, top_k=FUSION_TOP_K):
    score_map = {}
    doc_map = {}

    for ranked_docs in dense_ranked_lists:
        for rank, d in enumerate(ranked_docs, start=1):
            k = doc_key(d)
            score_map[k] = score_map.get(k, 0.0) + 1.0 / (RRF_K + rank)
            doc_map[k] = d

    for ranked_docs in sparse_ranked_lists:
        for rank, d in enumerate(ranked_docs, start=1):
            k = doc_key(d)
            score_map[k] = score_map.get(k, 0.0) + 1.0 / (RRF_K + rank)
            doc_map[k] = d

    ranked = sorted(score_map.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [doc_map[k] for k, _ in ranked]


def docs_to_context(docs, include_metadata=True):
    lines = []
    for i, d in enumerate(docs, start=1):
        meta = getattr(d, "metadata", {}) or {}
        source = meta.get("source", "unknown")
        page = meta.get("page", "unknown")
        text = (getattr(d, "page_content", "") or "").strip()
        if include_metadata:
            lines.append(f"[Doc {i}] source={source}, page={page}\n{text}")
        else:
            lines.append(f"[Doc {i}]\n{text}")
    return "\n\n---\n\n".join(lines)


def safe_json(text: str, fallback: dict):
    try:
        return json.loads(text)
    except Exception:
        cleaned = text.strip().replace("```json", "").replace("```", "")
        try:
            return json.loads(cleaned)
        except Exception:
            return fallback.copy()


def get_reranker():
    global cross_encoder
    if CrossEncoder is None:
        return None
    if cross_encoder is None:
        cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    return cross_encoder


# =====================================================
# 9. GRAPH STATE
# =====================================================
class AgentState(TypedDict):
    user_id: int
    user_input: str
    current_symptoms: dict
    conversation_summary: list
    last_followup_question: Optional[str]
    followup_count: int
    doctor_output: str
    final_reply: str
    session_active: bool

    optimized_query: str
    refined_query: str
    active_query: str
    query_variants: list
    query_generation_debug: dict

    retrieved_dense_docs: list
    retrieved_sparse_docs: list
    fused_docs: list
    retrieved_docs: list

    pre_critic_score: int
    pre_critic_passed: bool
    pre_critic_feedback: str
    pre_critic_retry_count: int
    retry_exhausted: bool

    reranked_docs: list
    reranker_scores: list
    selected_docs: list

    final_context: str
    retrieval_trace: dict


# =====================================================
# 10. GRAPH NODES
# =====================================================
def get_user_input_node(state: AgentState) -> dict:
    if state.get("last_followup_question"):
        print(f"\nFollow-up: {state['last_followup_question']}")
    else:
        print("\nCompounder Agent: Tell me how you've been feeling lately.")

    user_input: str = interrupt("Waiting for user input...")

    if user_input.strip().lower() in ["exit", "quit", "bye"]:
        print("\nSession ended. Take care.")
        return {"session_active": False, "user_input": ""}

    return {"session_active": True, "user_input": user_input.strip()}


def compounder_agent(state: AgentState) -> dict:
    user_input = state["user_input"]
    last_q = state.get("last_followup_question")
    conv_summary = state.get("conversation_summary", [])

    compounder_input = user_input
    if last_q:
        compounder_input = f"Previous question: {last_q}\nAnswer: {user_input}\n"
    if conv_summary:
        compounder_input += "Conversation summary so far: " + "; ".join(conv_summary) + "\n"

    extracted_full_json = extract_symptoms(compounder_input)
    new_structured_data = extracted_full_json.get("structured_fields", {})

    updated_symptoms = merge_symptoms(state["current_symptoms"], new_structured_data)
    updated_summary = update_conversation_summary(conv_summary, new_structured_data)

    save_symptoms(state["user_id"], updated_symptoms)

    print("\nUpdated State Symptoms (Internal):")
    print(json.dumps(updated_symptoms, indent=2))

    return {
        "current_symptoms": updated_symptoms,
        "conversation_summary": updated_summary,
    }


def followup_agent(state: AgentState) -> dict:
    followup_count = state.get("followup_count", 0)

    if followup_count < MAX_FOLLOWUPS:
        followup = get_dynamic_followup(
            state["current_symptoms"],
            state.get("last_followup_question"),
            "; ".join(state.get("conversation_summary", []))
        )
    else:
        followup = None

    if followup:
        return {
            "last_followup_question": followup,
            "followup_count": followup_count + 1,
        }

    print("\nEnough information collected.")
    return {
        "last_followup_question": None,
        "followup_count": followup_count,
    }


def query_optimizer_node(state: AgentState) -> dict:
    generated = generate_query_variants(
        symptoms=state["current_symptoms"],
        user_text=state.get("user_input", ""),
        conversation_summary=state.get("conversation_summary", []),
    )

    refined = (state.get("refined_query") or "").strip()
    variants = generated["query_variants"]
    if refined:
        variants = normalize_query_list([refined] + variants, QUERY_VARIANT_LIMIT)

    active = variants[0] if variants else generated["primary_query"]

    print(f"\nQuery Optimizer -> active={active}")
    print(f"Query Variants -> {variants}")

    return {
        "optimized_query": generated["primary_query"],
        "active_query": active,
        "query_variants": variants,
        "query_generation_debug": generated.get("query_generation_raw", {}),
        "retry_exhausted": False,
    }


def hybrid_retrieval_node(state: AgentState) -> dict:
    t0 = perf_counter()

    queries = state.get("query_variants") or []
    if not queries:
        fallback_q = state.get("active_query") or state.get("optimized_query") or ""
        queries = [fallback_q]
    queries = normalize_query_list(queries, QUERY_VARIANT_LIMIT)

    cache_key = " || ".join(queries)
    if cache_key in retrieval_cache_hybrid:
        payload = retrieval_cache_hybrid[cache_key].copy()
        payload["retrieval_trace"] = dict(payload.get("retrieval_trace", {}))
        payload["retrieval_trace"]["cache_hit"] = True
        print("\nHybrid Retrieval -> cache hit")
        return payload

    dense_ranked_lists = []
    sparse_ranked_lists = []
    per_query_stats = []

    all_dense = []
    all_sparse = []

    for q in queries:
        dense_docs = retrieve_dense(q)
        sparse_docs = retrieve_sparse(q)

        dense_ranked_lists.append(dense_docs)
        sparse_ranked_lists.append(sparse_docs)

        all_dense.extend(dense_docs)
        all_sparse.extend(sparse_docs)

        per_query_stats.append({
            "query": q,
            "dense_count": len(dense_docs),
            "sparse_count": len(sparse_docs),
        })

    fused_docs = reciprocal_rank_fusion_multi(
        dense_ranked_lists=dense_ranked_lists,
        sparse_ranked_lists=sparse_ranked_lists,
        top_k=FUSION_TOP_K,
    )

    payload = {
        "retrieved_dense_docs": all_dense,
        "retrieved_sparse_docs": all_sparse,
        "fused_docs": fused_docs,
        "retrieved_docs": fused_docs,
        "retrieval_trace": {
            "queries": queries,
            "query_count": len(queries),
            "per_query": per_query_stats,
            "dense_total": len(all_dense),
            "sparse_total": len(all_sparse),
            "fused_count": len(fused_docs),
            "latency_ms": int((perf_counter() - t0) * 1000),
            "cache_hit": False,
            "sparse_enabled": bm25 is not None,
        },
    }
    retrieval_cache_hybrid[cache_key] = payload

    print(f"\nHybrid Retrieval -> queries={len(queries)} fused={len(fused_docs)}")
    return payload


def pre_critic_node(state: AgentState) -> dict:
    symptoms_json = json.dumps(state["current_symptoms"])
    context_text = docs_to_context(state.get("retrieved_docs", []), include_metadata=True)

    raw = pre_critic_chain.invoke({
        "symptoms": symptoms_json,
        "context": context_text,
    })

    parsed = safe_json(raw, {
        "score": 0,
        "sufficient": False,
        "reason": "Invalid critic JSON",
        "refinement_query": "",
    })

    score = int(parsed.get("score", 0))
    passed = bool(parsed.get("sufficient", False)) and score >= PRE_CRITIC_PASS_SCORE

    print(f"\nPre-Critic -> score={score}, passed={passed}")
    return {
        "pre_critic_score": score,
        "pre_critic_passed": passed,
        "pre_critic_feedback": json.dumps(parsed),
    }


def pre_critic_retry_node(state: AgentState) -> dict:
    retries = state.get("pre_critic_retry_count", 0)
    if retries >= PRE_CRITIC_MAX_RETRIES:
        print("\nPre-Critic Retry -> max retries reached, continuing")
        return {
            "retry_exhausted": True,
            "pre_critic_passed": True,
        }

    feedback = safe_json(state.get("pre_critic_feedback", "{}"), {})
    refined = (feedback.get("refinement_query") or "").strip()
    if not refined:
        refined = state.get("active_query") or state.get("optimized_query") or ""

    existing_variants = state.get("query_variants") or []
    new_variants = normalize_query_list([refined] + existing_variants, QUERY_VARIANT_LIMIT)

    print(f"\nPre-Critic Retry -> retry={retries + 1}, refined_query={refined}")
    return {
        "refined_query": refined,
        "active_query": new_variants[0] if new_variants else refined,
        "query_variants": new_variants,
        "pre_critic_retry_count": retries + 1,
    }


def reranker_node(state: AgentState) -> dict:
    docs = state.get("retrieved_docs", [])
    query = state.get("active_query") or state.get("optimized_query") or ""

    if not docs:
        return {
            "reranked_docs": [],
            "reranker_scores": [],
            "selected_docs": [],
            "final_context": "",
        }

    try:
        reranker = get_reranker()
        if reranker is None:
            raise RuntimeError("CrossEncoder unavailable")
        pairs = [[query, d.page_content] for d in docs]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    except Exception:
        ranked = [(d, 0.0) for d in docs]

    top = ranked[:FINAL_CONTEXT_TOP_K]
    selected_docs = [d for d, _ in top]

    score_rows = [
        {
            "score": float(s),
            "source": (d.metadata or {}).get("source", "unknown"),
            "page": (d.metadata or {}).get("page", "unknown"),
        }
        for d, s in top
    ]

    context = docs_to_context(selected_docs, include_metadata=True)
    trace = dict(state.get("retrieval_trace", {}))
    trace["reranked_top_k"] = len(selected_docs)
    trace["reranker_enabled"] = CrossEncoder is not None

    print(f"\nReranker -> selected={len(selected_docs)}")
    return {
        "reranked_docs": [d for d, _ in ranked],
        "reranker_scores": score_rows,
        "selected_docs": selected_docs,
        "final_context": context,
        "retrieval_trace": trace,
    }


def doctor_agent(state: AgentState) -> dict:
    print("\nDoctor Agent: Analysing symptoms...")

    symptoms_json = json.dumps(state["current_symptoms"])
    history = get_user_history(state["user_id"])

    trace = json.dumps(state.get("retrieval_trace", {}))
    context = state.get("final_context", "")
    enriched_context = f"{context}\n\n[Retrieval metadata]\n{trace}"

    result = doctor_generation_chain.invoke({
        "symptoms": symptoms_json,
        "history": history,
        "context": enriched_context,
    })

    return {"doctor_output": result}


def summary_agent(state: AgentState) -> dict:
    final_reply = synopsis_chain.invoke({"doctor_output": state["doctor_output"]}).strip()

    print("\nAssistant:", final_reply)
    print("-" * 60)

    return {"final_reply": final_reply}


# =====================================================
# 11. CONDITIONAL EDGE ROUTER
# =====================================================
def route_after_followup(state: AgentState) -> str:
    if not state.get("session_active", True):
        return "end"
    if state.get("last_followup_question"):
        return "get_user_input"
    return "query_optimizer_node"


def route_after_input(state: AgentState) -> str:
    if not state.get("session_active", True):
        return "end"
    return "compounder_agent"


def route_after_pre_critic(state: AgentState) -> str:
    if state.get("pre_critic_passed", False):
        return "reranker_node"
    return "pre_critic_retry_node"


def route_after_retry(state: AgentState) -> str:
    if state.get("retry_exhausted", False):
        return "reranker_node"
    return "hybrid_retrieval_node"


# =====================================================
# 12. BUILD THE LANGGRAPH WORKFLOW
# =====================================================
def build_graph() -> StateGraph:
    builder = StateGraph(AgentState)

    # Register nodes
    builder.add_node("get_user_input", get_user_input_node)
    builder.add_node("compounder_agent", compounder_agent)
    builder.add_node("followup_agent", followup_agent)
    builder.add_node("query_optimizer_node", query_optimizer_node)
    builder.add_node("hybrid_retrieval_node", hybrid_retrieval_node)
    builder.add_node("pre_critic_node", pre_critic_node)
    builder.add_node("pre_critic_retry_node", pre_critic_retry_node)
    builder.add_node("reranker_node", reranker_node)
    builder.add_node("doctor_agent", doctor_agent)
    builder.add_node("summary_agent", summary_agent)

    # Edges
    builder.add_edge(START, "get_user_input")

    builder.add_conditional_edges(
        "get_user_input",
        route_after_input,
        {"compounder_agent": "compounder_agent", "end": END},
    )

    builder.add_edge("compounder_agent", "followup_agent")

    builder.add_conditional_edges(
        "followup_agent",
        route_after_followup,
        {
            "get_user_input": "get_user_input",
            "query_optimizer_node": "query_optimizer_node",
            "end": END,
        },
    )

    builder.add_edge("query_optimizer_node", "hybrid_retrieval_node")
    builder.add_edge("hybrid_retrieval_node", "pre_critic_node")

    builder.add_conditional_edges(
        "pre_critic_node",
        route_after_pre_critic,
        {
            "reranker_node": "reranker_node",
            "pre_critic_retry_node": "pre_critic_retry_node",
        },
    )

    builder.add_conditional_edges(
        "pre_critic_retry_node",
        route_after_retry,
        {
            "hybrid_retrieval_node": "hybrid_retrieval_node",
            "reranker_node": "reranker_node",
        },
    )

    builder.add_edge("reranker_node", "doctor_agent")
    builder.add_edge("doctor_agent", "summary_agent")
    builder.add_edge("summary_agent", END)

    return builder.compile(checkpointer=MemorySaver())


# =====================================================
# 13. MAIN RUNNER
# =====================================================
def run_session():
    print("\nMental Health Assistant (LangGraph Edition)")
    print("This assistant will gently ask questions to understand how you feel.")
    print("Type 'exit' anytime to stop.\n")

    while True:
        try:
            user_id = int(input("Enter your user ID: "))
            break
        except ValueError:
            print("Please enter a valid numeric user ID.\n")

    graph = build_graph()
    config = {"configurable": {"thread_id": str(user_id)}}

    initial_state: AgentState = {
        "user_id": user_id,
        "user_input": "",
        "current_symptoms": {field: "unknown" for field in SYMPTOM_FIELDS},
        "conversation_summary": [],
        "last_followup_question": None,
        "followup_count": 0,
        "doctor_output": "",
        "final_reply": "",
        "session_active": True,

        "optimized_query": "",
        "refined_query": "",
        "active_query": "",
        "query_variants": [],
        "query_generation_debug": {},

        "retrieved_dense_docs": [],
        "retrieved_sparse_docs": [],
        "fused_docs": [],
        "retrieved_docs": [],

        "pre_critic_score": 0,
        "pre_critic_passed": False,
        "pre_critic_feedback": "",
        "pre_critic_retry_count": 0,
        "retry_exhausted": False,

        "reranked_docs": [],
        "reranker_scores": [],
        "selected_docs": [],

        "final_context": "",
        "retrieval_trace": {},
    }

    # First run: graph will hit interrupt() inside get_user_input_node
    for _ in graph.stream(initial_state, config, stream_mode="values"):
        pass

    # Resume loop: feed user replies until the graph reaches END
    while True:
        snapshot = graph.get_state(config)

        if not snapshot.next:
            break

        user_input = input("\nYou: ").strip()
        for _ in graph.stream(Command(resume=user_input), config, stream_mode="values"):
            pass


# =====================================================
# ENTRY POINT
# =====================================================
if __name__ == "__main__":
    run_session()

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.

Mental Health Assistant (LangGraph Edition)
This assistant will gently ask questions to understand how you feel.
Type 'exit' anytime to stop.


Compounder Agent: Tell me how you've been feeling lately.

Compounder Agent: Tell me how you've been feeling lately.

Updated State Symptoms (Internal):
{
  "mood": "sad",
  "anxiety": "unknown",
  "stress": "unknown",
  "sleep": "unknown",
  "energy": "unknown

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]


Reranker -> selected=4

Doctor Agent: Analysing symptoms...

Assistant: It sounds like you've been going through a tough time lately, and I want to acknowledge that. Here’s a quick overview of what you might be experiencing:

- **Sadness**: A common feeling in depression that can affect your outlook.
- **Low Energy**: Fatigue can make daily tasks feel hard.
- **Sleep Issues**: Trouble sleeping can worsen your mood.
- **Decreased Appetite**: Changes in eating habits are often seen in depression.

Given the severity of your symptoms, it’s really important to consider talking to a mental health professional. They can offer support and help you find ways to feel better. If you're having thoughts of self-harm or suicide, please seek help right away. Remember, you're not alone, and there are people who care and want to help!
------------------------------------------------------------


In [2]:
import os

# Create the 'mentaldocs' directory if it doesn't already exist
os.makedirs('mentaldocs', exist_ok=True)
print("Directory 'mentaldocs' ensured to exist.")

Directory 'mentaldocs' ensured to exist.


In [3]:
import sys
print(sys.executable)
print(sys.version)

/usr/local/bin/python3
3.13.1 (v3.13.1:06714517797, Dec  3 2024, 14:00:22) [Clang 15.0.0 (clang-1500.3.9.4)]
